In [1]:
import time
start_time = time.perf_counter()
print('Done!')
import re

Done!


In [2]:
the_directory = 'output/demo'
ending = 'XXXTHISENDSHEREXXX'

In [3]:
from kfunc_filter import should_filter_function
from ollama import chat
from collections import defaultdict, deque
import networkx as nx
import matplotlib.pyplot as plt
import os
import scipy
import re
import time

start_time = time.perf_counter()
model = 'qwen3:32b'
role = 'user'
num_ctx = 14336
the_directory = 'output/demo'
demo_profile = "./demo-profile-sysclose.txt"
search_dir = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
kernel_cg_path = "./pruned_callgraph.graphml"
k_cg = nx.read_graphml(kernel_cg_path)
k_cg.remove_edges_from(nx.selfloop_edges(k_cg))
kall_functions = [node for node in k_cg.nodes if not should_filter_function(node)]

def gen_subgraph(static_graph, sys_entry_function, function_set, remove_broken_edges=False):
    function_set = [f for f in function_set if not should_filter_function(f)]
    subgraph_nodes = set(function_set)
    if remove_broken_edges:
        reachable_nodes = nx.descendants(static_graph, sys_entry_function)
        reachable_nodes.add(sys_entry_function)
        subgraph_nodes = subgraph_nodes.intersection(reachable_nodes)
    subgraph = static_graph.subgraph(subgraph_nodes).copy()
    return subgraph

def outedges_callgraph(graph, function_name, do_filter=True):
    if function_name not in graph:
        print(f"Function {function_name} does not exist in the graph.")
        return []
    out_edges = []
    for _, target, edge_data in graph.out_edges(function_name, data=True):
        if do_filter and should_filter_function(target):
            continue
        out_edges.append((function_name, target, edge_data))
    return out_edges

def export_graph_as_markdown_tree(graph):
    def dfs(node, indent=0, visited=set()):
        lines = []
        prefix = "  " * indent + "- " + node
        lines.append(prefix)
        visited.add(node)
        for _, neighbor in graph.out_edges(node):
            if neighbor not in visited:
                lines.extend(dfs(neighbor, indent + 1, visited))
        return lines
    roots = [n for n in graph.nodes if graph.in_degree(n) == 0]
    all_lines = []
    for root in roots:
        all_lines.extend(dfs(root, indent=0, visited=set()))
    return "\n".join(all_lines)

def find_function_source(root_dir, function_name, exts=('.c',)):
    sig_re = re.compile(
        rf'''^[ \t]*
            (?:[A-Za-z_]\w*  # return type or storage-class (e.g. "static", "inline", "void", "struct foo")
               (?:\s+[\w\*\s]+)*)?  # allow pointers and multiple words
            \b{re.escape(function_name)}\s*\(  # the function name
        ''',
        re.VERBOSE
    )
    def scan_file(path):
        try:
            f = open(path, 'r', errors='ignore')
        except PermissionError:
            return None
        with f:
            collecting = False
            brace_count = 0
            buffer = []
            in_block = False
            for raw in f:
                line = raw
                if in_block:
                    end = line.find('*/')
                    if end >= 0:
                        in_block = False
                        line = line[end+2:]
                    else:
                        continue
                start = line.find('/*')
                if start >= 0:
                    end = line.find('*/', start+2)
                    if end >= 0:
                        line = line[:start] + line[end+2:]
                    else:
                        in_block = True
                        line = line[:start]
                clean = line.split('//', 1)[0]
                if clean.rstrip().endswith('\\'):
                    continue
                if not collecting:
                    if sig_re.match(clean):
                        sig_raw = [raw]
                        sig_clean = [clean]
                        bal = clean.count('(') - clean.count(')')
                        while bal > 0:
                            nxt_raw = f.readline()
                            if not nxt_raw:
                                break
                            nxt = nxt_raw
                            if in_block:
                                end = nxt.find('*/')
                                if end >= 0:
                                    in_block = False
                                    nxt = nxt[end+2:]
                                else:
                                    continue
                            st = nxt.find('/*')
                            if st >= 0:
                                ed = nxt.find('*/', st+2)
                                if ed >= 0:
                                    nxt = nxt[:st] + nxt[ed+2:]
                                else:
                                    in_block = True
                                    nxt = nxt[:st]
                            nxt_clean = nxt.split('//',1)[0]
                            sig_raw.append(nxt_raw)
                            sig_clean.append(nxt_clean)
                            bal += nxt_clean.count('(') - nxt_clean.count(')')
                        if ''.join(sig_clean).strip().endswith(';'):
                            continue
                        buffer = sig_raw.copy()
                        if '{' in sig_clean[-1]:
                            collecting = True
                            brace_count = sig_clean[-1].count('{') - sig_clean[-1].count('}')
                        else:
                            for body_raw in f:
                                body = body_raw
                                if in_block:
                                    end = body.find('*/')
                                    if end >= 0:
                                        in_block = False
                                        body = body[end+2:]
                                    else:
                                        continue
                                st = body.find('/*')
                                if st >= 0:
                                    ed = body.find('*/', st+2)
                                    if ed >= 0:
                                        body = body[:st] + body[ed+2:]
                                    else:
                                        in_block = True
                                        body = body[:st]
                                body_clean = body.split('//',1)[0]
                                buffer.append(body_raw)
                                if '{' in body_clean:
                                    collecting = True
                                    brace_count = (
                                        body_clean.count('{') - 
                                        body_clean.count('}')
                                    )
                                    break
                        if not collecting:
                            buffer = []
                else:
                    buffer.append(raw)
                    tmp = raw
                    if in_block:
                        end = tmp.find('*/')
                        if end >= 0:
                            in_block = False
                            tmp = tmp[end+2:]
                        else:
                            continue
                    st = tmp.find('/*')
                    if st >= 0:
                        ed = tmp.find('*/', st+2)
                        if ed >= 0:
                            tmp = tmp[:st] + tmp[ed+2:]
                        else:
                            in_block = True
                            tmp = tmp[:st]
                    tmp_clean = tmp.split('//',1)[0]

                    brace_count += tmp_clean.count('{') - tmp_clean.count('}')
                    if brace_count == 0:
                        return ''.join(buffer)
        return None
    def recurse(path):
        try:
            entries = os.scandir(path)
        except PermissionError:
            return None
        with entries:
            for e in entries:
                name = e.name.lower()
                if name.endswith(exts):
                    try:
                        if e.is_file(follow_symlinks=False):
                            src = scan_file(e.path)
                            if src:
                                return src
                    except PermissionError:
                        pass
                try:
                    if e.is_dir(follow_symlinks=False):
                        found = recurse(e.path)
                        if found:
                            return found
                except PermissionError:
                    pass
        return None
    return recurse(root_dir)

def find_call_chain(src, dst, calls):
    visited = set([src])
    queue = deque([[src]])
    while queue:
        path = queue.popleft()
        last = path[-1]
        if last == dst:
            return path
        for callee in calls.get(last, []):
            if callee not in visited:
                visited.add(callee)
                queue.append(path + [callee])
    return None

def get_in_between_functions(calls, src, dst):
    chain = find_call_chain(src, dst, calls)
    if chain and len(chain) > 2:
        return chain[1:-1]
    return []

demo_profiled_functions = set()

with open(demo_profile, "r") as f:
    lines = f.readlines()
    for line in lines:
        func_name = line.strip()
        if func_name in kall_functions and not should_filter_function(func_name):
            demo_profiled_functions.add(func_name)

subgraph = gen_subgraph(k_cg, "__x64_sys_close", demo_profiled_functions)
subgraph_start = subgraph.copy()

print('Done!')

Done!


In [4]:
md_file    = 'the_start_markdown.txt'
src_file   = 'the_functions_all.txt'
out_file   = 'in_between_list.txt'

# 1) Load seluruh source block dari the_functions_all.txt
with open(f"{the_directory}/{src_file}") as f:
    content = f.read()

ending = 'XXXTHISENDSHEREXXX'
pattern = rf"Source Code for\s+([\w_]+)\s*:\s*\n(.*?)(?={re.escape(ending)})"
all_blocks = dict(re.findall(pattern, content, flags=re.DOTALL))



In [5]:
is_there_something_new = True
has_been_checked = []
the_output = f'{the_directory}/the_real_llm.txt'

with open(the_output, 'w', encoding='utf-8') as f:
    f.write('')

function_sources = {}
node_set = set(subgraph.nodes)
start = True

while (is_there_something_new):
    new_nodes = list(set(subgraph.nodes) - node_set)
    if start:
        new_nodes = list(node_set)
        start = False
    for node in new_nodes:
        if node in all_blocks:
            function_sources[node] = all_blocks[node]
        else:
            # kalau mau, bisa set None atau skip
            function_sources[node] = None
    is_there_something_new = False
    topo_sorted = list(nx.topological_sort(subgraph))
    reverse_topo_sorted = list(reversed(topo_sorted))
    try_this = defaultdict(list)
    for i in reverse_topo_sorted:
        out_edges = outedges_callgraph(k_cg, i)
        for src, dst, edge_data in out_edges:
            try_this[src].append(dst)
    markdown_output = export_graph_as_markdown_tree(subgraph)
    for src, targets in try_this.items():
        for dst in targets:
            if subgraph.has_edge(src, dst):
                continue
            if dst in has_been_checked:
                continue
            else:
                has_been_checked.append(dst)
            dst_code = all_blocks.get(dst)
            prompt_text = f'''You are a Linux security expert analyzing kernel call-graph edges. The following historical dynamic execution path was observed:
{markdown_output}
Source code for {dst}:
{dst_code}
'''
            for node, code in function_sources.items():
                prompt_text += f"Source code for {node}:\n{code}"
            #in_between_functions = get_in_between_functions(k_cg, src, dst)
            #for fn in in_between_functions:
                #code = find_function_source(search_dir, fn)
                #prompt_text += f"Source code for intermediate {fn}:\n{code}"
            prompt_text += f'From a security-engineering standpoint, is it reasonable to expect that execution of {src} will reach {dst}? Provide a concise technical justification for your position, then state your conclusion in exactly this format: {{Your justification}}\\nFINAL ANSWER -> YES/NO'
            prompt = {
                'role': 'user',
                'content': prompt_text
            }
            with open(the_output, 'a', encoding='utf-8') as file:
                file.write(f'🟢🟢🟢PROMPT🟢🟢🟢: {prompt_text}\n')
            response = chat(model=model, messages=[{'role': role, 'content': prompt_text}], options={'num_ctx': num_ctx})
            filtered_response = re.sub(r'<think>.*?</think>', '', response.message.content, flags=re.DOTALL).strip()
            filtered_response = re.sub(r'[ \t]+$', '', filtered_response, flags=re.MULTILINE)
            with open(the_output, 'a', encoding='utf-8') as file:
                file.write(f'🟢🟢🟢RESPONSE🟢🟢🟢: {filtered_response}\n')
            if "FINAL ANSWER -> YES" in filtered_response.upper():
                is_there_something_new = True
                with open(the_output, 'a', encoding='utf-8') as file:
                    file.write(f'🟢 Upgrading edge {src} -> {dst} to subgraph (LLM confirmed)')
                    file.write('\n')
                subgraph.add_edge(src, dst, inferred=True)
            with open(the_output, 'a', encoding='utf-8') as file:
                file.write('\n\n\n')
    if is_there_something_new:
        with open(the_output, 'a', encoding='utf-8') as file:
            file.write('🔴🔴🔴NEW SUBGRAPH!🔴🔴🔴')
            file.write('\n')
            file.write('🔴🔴🔴NEW SUBGRAPH!🔴🔴🔴')
            file.write('\n')
            file.write('🔴🔴🔴NEW SUBGRAPH!🔴🔴🔴')
            file.write('\n')
            file.write('\n\n\n\n')
            file.write('\n')

with open(the_output, 'a', encoding='utf-8') as file:
    file.write('THE END!')

##################################################
markdown_output = export_graph_as_markdown_tree(subgraph)
the_markdown = f'{the_directory}/the_real_markdown.txt'

with open(the_markdown, 'w', encoding='utf-8') as file:
    file.write(markdown_output)

##################################################
end_time = time.perf_counter()
elapsed = end_time - start_time
hours   = int(elapsed // 3600)
minutes = int((elapsed % 3600) // 60)
seconds = elapsed % 60

print(f"Elapsed time: {hours}h {minutes}m {seconds:.6f}s")
print('Done!')

Elapsed time: 0h 45m 24.637572s
Done!
